[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/weighted_dual/weighted_dual.ipynb)

# Weighted Polar Duals on the Sphere

This example computes the polar dual of a polyhedron about the unit sphere and tracks how its dual vertices move as the primal vertices slide along fixed rays from the centre. Projected radially onto the sphere, the polyhedron's faces form a spherical triangulation while the poles of those faces form the vertices of the dual complex. Changing a vertex's distance from the centre leaves its radial projection fixed while tilting the incident face planes and shifting their poles across the sphere. Moving the vertices radially first drives a breathing dual complex on a fixed triangulation, and then untangles a folded triangulation by finding the nearest vertex weights that make every dual edge positive.

The term ***simplex gradient*** here means the corner plane `edge_planes / (edge_planes & corners)` that pairs to one with its own corner and to zero with the two opposite corners. Summing the three corner simplex gradients of a triangle forms the face plane `plane & vertex == 1`, and polarity about the unit sphere maps that plane to its pole `gradients.sum(axis=0).dual()`. Comparing `(walls & duals) * (walls & corners)` tests whether each pole projects inside its triangle without choosing an ambient handedness, while pulling poles back across interior edges with `d[1].adjugate()` measures signed dual edge lengths. Parameterizing each vertex along its ray by inverse distance makes both the dihedral folds and the weighted face planes `gradients * corner_weights` linear in the weights, reducing untangling to a convex linear program.

In Gibbs–Heaviside vector and discrete exterior calculus notation, each face pole is written with cross and scalar triple products as $\mathbf p_{ijk} = (\mathbf v_i \times \mathbf v_j + \mathbf v_j \times \mathbf v_k + \mathbf v_k \times \mathbf v_i) / (\mathbf v_i \cdot (\mathbf v_j \times \mathbf v_k))$, and signing the dual step across an edge $(\mathbf v_a, \mathbf v_b)$ for the diagonal Hodge star $\star_1$ projects $\mathbf p_R - \mathbf p_L$ onto the oriented cross product $\mathbf v_a \times \mathbf v_b$. Varying radial weights corresponds there to lifting spherical sites to power-diagram weights $w_i$ and tracking orthocentres of weighted Delaunay triangles via separate determinant inequalities on adjacent four-tuples.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from examples.surfaces.weighted_dual import numerical, render, scenarios
from examples.util.animation import save_figure, show_figure, show_states
from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from numga.topology import Topology

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# Points and ray directions from the centre are intrinsic antivectors.
Point = ga.gatype.antivector()                      # Antivector-

## Faces and poles

Dividing each opposite-edge plane `opposite_corners[0] & opposite_corners[1]` by its pairing with the corner `edge_planes & corners` cancels edge orientation and yields the three corner simplex gradients of each triangle. Their sum pairs to one with all three corners, so `gradients.sum(axis=0).dual()` gives the face pole, and `(walls & duals) * (walls & corners)` is positive on all three sides whenever that pole projects inside the spherical triangle. SciPy supplies the initial 3D convex hull topology outside the algebra.

In [ ]:
n_points = 32
jitter = 0.03
direction_seed = 0

In [ ]:
def spiral_directions(n_points: int, jitter: float, seed: int) -> Point:
    """Unit ray directions (`Point-[V]`) follow a Fibonacci spiral perturbed by Gaussian angular jitter."""
    turns = np.arange(n_points) + 0.5                                          # [n_points]
    polar = np.arccos(1 - 2 * turns / n_points)                                # [n_points]
    azimuth = np.pi * (1 + 5**0.5) * turns                                     # [n_points]
    # Turn the north pole toward +x by the polar angle and around the z axis by the azimuth.
    polar_rotors = (mv.zx * (-polar / 2)).exp()                                # [n_points] Rotor+
    azimuth_rotors = (mv.xy * (-azimuth / 2)).exp()                            # [n_points] Rotor+
    spiral = (azimuth_rotors * polar_rotors) >> mv.z.dual()                    # [n_points] Point-
    nudges = mv.vector(np.random.default_rng(seed).normal(size=(n_points, 3))).dual()  # [n_points] Point-
    return (spiral + jitter * nudges).normalized().field()                     # Point-[V]


directions = spiral_directions(n_points, jitter, direction_seed)               # Point-[V]
sphere_topology = numerical.convex_hull(directions)

In [ ]:
def face_poles(topology: Topology, points: Point) -> Point:
    """Each face plane sums its three corner simplex gradients, and its dual about the unit sphere gives the face pole (`Point-[F]`)."""
    corners = topology.incidence[2, 0] * points                                # [3] Point-[F]
    opposite_corners = topology.off[2, 0] * points                             # [2, 3] Point-[F]
    edge_planes = opposite_corners[0] & opposite_corners[1]                    # [3] Plane-[F]
    gradients = edge_planes / (edge_planes & corners)                          # [3] Plane+[F]
    return gradients.sum(axis=0).dual()                                        # Point-[F]


def pole_sides(topology: Topology, points: Point, duals: Point) -> Scalar:
    """Pairings with the three central edge planes (`[3] Scalar+[F]`) are all positive when each face pole projects inside its triangle."""
    corners = topology.incidence[2, 0] * points                                # [3] Point-[F]
    opposite_corners = topology.off[2, 0] * points                             # [2, 3] Point-[F]
    walls = opposite_corners[0] & opposite_corners[1]                          # [3] Plane-[F]
    return (walls & duals) * (walls & corners)                                 # [3] Scalar+[F]


sphere_poles = face_poles(sphere_topology, directions)                         # Point-[F]
sphere_sides = pole_sides(sphere_topology, directions, sphere_poles)           # [3] Scalar+[F]
show_figure(render.complex_figure(sphere_topology, directions, sphere_poles))

The left panel renders the spatial polyhedron in light blue with primal edges in dark blue, vertices along grey radial segments, and front-facing polar dual edges in orange. The right panel inks the radial projections of both complexes as great-circle arcs on the front hemisphere, where unit-sphere vertices place every dual vertex at its triangle's spherical circumcentre.

## Sliding vertices along fixed rays

Holding the spherical triangulation fixed while each vertex oscillates radially along its own ray leaves every primal arc on the sphere stationary. Tilting the spatial triangles shifts their poles across the sphere, moving the dual vertices away from the spherical circumcentres and occasionally outside their triangles while preserving exact polar incidence in space.

In [ ]:
phase_seed = 3
amplitude = 0.06
n_frames = 96
duration_ms = 40

In [ ]:
def breath_states(
    topology: Topology, directions: Point, amplitude: float, phases: np.ndarray, n_frames: int
) -> Iterator[tuple[Topology, Point, Point]]:
    """Yields `(topology, points, duals)` across `n_frames` as vertices oscillate radially along fixed `directions` (`Point-[V]`)."""
    for angle in np.linspace(0, 2 * np.pi, n_frames, endpoint=False):
        radii = mv.scalar((1 + amplitude * np.sin(angle + phases))[:, None]).field()  # Scalar+[V]
        points = directions * radii                                            # Point-[V]
        yield topology, points, face_poles(topology, points)


phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, n_points)     # [n_points]
breath_history = show_states(
    breath_states(sphere_topology, directions, amplitude, phases, n_frames),
    render.complex_frames,
    name="weighted_dual_breath",
    duration_ms=duration_ms,
)

Vertices inside the unit sphere shade toward blue and vertices outside shade toward red, driving the orange dual complex across the sphere while the blue primal triangulation remains fixed.

In [ ]:
V_sphere, E_sphere, F_sphere = sphere_topology.n_elements
centred_counts = np.array([
    (pole_sides(topology, points, duals) > 0).all(axis=0).sum() for topology, points, duals in breath_history
])                                                                             # [n_frames]
print(f"well-centred faces over the breath {centred_counts.min()} to {centred_counts.max()} of {F_sphere}")
save_figure(render.complex_figure(*breath_history[0]), "weighted_dual");

## Weights that keep every dual edge positive

Taking the convex hull of vertices perturbed to random radii produces a distinct triangulation of the same ray directions. Resetting those vertices onto the unit sphere folds several interior edges inward and flips the adjacent dual steps negative, breaking positivity of dual-edge operators.

Along fixed rays, parameterizing each vertex by its inverse-distance weight makes both the four-corner dihedral fold determinants and the weighted face planes `gradients * corner_weights` linear in the weights. A linear program finds the widest feasible margin across all fold and triangle-containment inequalities, and a quadratic program selects the nearest weights retaining a fixed share of that margin so every crossed dual edge unfolds monotonically along the straight weight path.

In [ ]:
radius_spread, radius_seed = 0.06, 57
weight_bounds = (0.5, 2.0)
margin_share = 0.3
n_rescue_frames = 96

In [ ]:
def edge_folds(topology: Topology, directions: Point, weights: Scalar) -> Scalar:
    """Signed dihedral folds (`[...] Scalar+[E_interior]`) across interior edges vary linearly with the four incident vertex weights."""
    interior = topology.reading(topology.interior[1], 1)                       # Scalar+[E_interior <- E]
    # Gather each interior edge's endpoints together with the opposite corners of its left and right triangles.
    ends = interior * topology.incidence[1, 0]                                 # [2] Scalar+[E_interior <- V]
    opposites = topology.reading(topology.facet_opposites.T, 0)                # [2] Scalar+[E_interior <- V]
    tail, head = ends * directions                                             # Point-[E_interior], Point-[E_interior]
    left_corner, right_corner = opposites * directions                         # Point-[E_interior], Point-[E_interior]
    left_volume = tail & head & left_corner                                    # Scalar-[E_interior]
    volumes = stack([
        head & left_corner & right_corner,
        tail & left_corner & right_corner,
        tail & head & right_corner,
        left_volume,
    ])                                                                         # [4] Scalar-[E_interior]
    # Weight each three-ray volume with alternating signs so the four-point determinant stays linear in the weights.
    corner_weights = concatenate([ends * weights[..., None], opposites * weights[..., None]], axis=-1)  # [..., 4] Scalar+[E_interior]
    signs = np.array([-1, 1, -1, 1])                                           # [4]
    return (volumes * (corner_weights * signs)).sum(axis=-1) * left_volume     # [...] Scalar+[E_interior]


def pole_insides(topology: Topology, directions: Point, weights: Scalar) -> Scalar:
    """Signed distances (`[..., 3] Scalar+[F]`) of each face pole inside its three central edge planes vary linearly with `weights`."""
    corners = topology.incidence[2, 0] * directions                            # [3] Point-[F]
    opposite_corners = topology.off[2, 0] * directions                         # [2, 3] Point-[F]
    edge_planes = opposite_corners[0] & opposite_corners[1]                    # [3] Plane-[F]
    gradients = edge_planes / (edge_planes & corners)                          # [3] Plane+[F]
    duals = (gradients * (topology.incidence[2, 0] * weights[..., None])).sum(axis=-1).dual()  # [...] Point-[F]
    walls = edge_planes.normalized()                                           # [3] Plane-[F]
    corner_senses = walls & corners                                            # [3] Scalar-[F]
    return (walls & duals[..., None]) * corner_senses / corner_senses.antinorm()  # [..., 3] Scalar+[F]


random_radii = mv.scalar(
    np.exp(radius_spread * np.random.default_rng(radius_seed).normal(size=(n_points, 1)))
).field()                                                                      # Scalar+[V]
tangled_topology = numerical.convex_hull(directions * random_radii)
even_weights = mv.scalar(np.ones((n_points, 1))).field()                       # Scalar+[V]
unit_weights = mv.scalar(np.eye(n_points)[..., None]).field()                  # [n_points] Scalar+[V]
fold_columns = edge_folds(tangled_topology, directions, unit_weights)          # [n_points] Scalar+[E_interior]
inside_columns = pole_insides(tangled_topology, directions, unit_weights)      # [n_points, 3] Scalar+[F]
_, max_margin = numerical.widest_margin(fold_columns, inside_columns, weight_bounds)  # Scalar+[V], Scalar+
rescued_weights = numerical.nearest_weights(
    fold_columns, inside_columns, margin_share * max_margin, even_weights, weight_bounds
)                                                                              # Scalar+[V]

In [ ]:
def dual_lengths(topology: Topology, points: Point) -> Scalar:
    """Signed steps across interior edges (`Scalar+[E_interior]`) between adjacent face poles turn negative where dual edges cross."""
    interior = topology.reading(topology.interior[1], 1)                       # Scalar+[E_interior <- E]
    tails, heads = interior * (topology.incidence[1, 0] * points)              # Point-[E_interior], Point-[E_interior]
    wall = tails & heads                                                       # Plane-[E_interior]
    right_sense = wall & (topology.reading(topology.facet_opposites[:, 1], 0) * points)  # Scalar-[E_interior]
    # Pulling face poles back by d[1].adjugate() gives left pole minus right pole, negated to step from left to right.
    pole_steps = -(interior * (topology.d[1].adjugate() * face_poles(topology, points)))  # Point-[E_interior]
    return (wall & pole_steps) * right_sense / right_sense.antinorm()          # Scalar+[E_interior]


def rescue_states(
    topology: Topology, directions: Point, start_weights: Scalar, target_weights: Scalar, schedule: np.ndarray
) -> Iterator[tuple[Topology, Point, Point]]:
    """Yields `(topology, points, duals)` as vertex weights (`Scalar+[V]`) interpolate linearly from `start_weights` to `target_weights`."""
    for progress in schedule:
        weights = start_weights + float(progress) * (target_weights - start_weights)  # Scalar+[V]
        points = directions / weights                                          # Point-[V]
        yield topology, points, face_poles(topology, points)


initial_dual_lengths = dual_lengths(tangled_topology, directions)              # Scalar+[E_interior]
rescue_schedule = scenarios.closing_schedule(n_rescue_frames)                  # [n_rescue_frames]
rescued_history = show_states(
    rescue_states(tangled_topology, directions, even_weights, rescued_weights, rescue_schedule),
    render.complex_frames,
    name="weighted_dual_rescue",
    duration_ms=duration_ms,
)

In [ ]:
V_tangled, E_tangled, F_tangled = tangled_topology.n_elements
final_dual_lengths = dual_lengths(tangled_topology, rescued_history[-1][1])    # Scalar+[E_interior]
for label, lengths, state in (
    ("at the start", initial_dual_lengths, rescued_history[0]),
    ("rescued", final_dual_lengths, rescued_history[-1]),
):
    print(
        f"{label:13s} negative dual edges {(lengths < 0).sum()} of {E_tangled},"
        f" well-centred faces {(pole_sides(*state) > 0).all(axis=0).sum()} of {F_tangled}"
    )
print(f"widest margin of the conditions {float(max_margin.to_array()):.4f}")

In [ ]:
# checks
assert (((sphere_poles.dual_inverse() & (sphere_topology.incidence[2, 0] * directions)) - 1).abs() < 1e-10).all()
assert (dual_lengths(sphere_topology, directions) > 0).all()
assert (sphere_sides > 0).all(axis=0).sum() > 0
for topology, points, duals in breath_history:
    # Every face plane is the inverse dual of its pole and pairs to one with all three corners.
    assert (((duals.dual_inverse() & (topology.incidence[2, 0] * points)) - 1).abs() < 1e-10).all()

# Negative dual lengths at unit weights match inward dihedral folds and uncross monotonically to zero.
crossed_counts = [(dual_lengths(tangled_topology, points) < 0).sum() for _, points, _ in rescued_history]
assert np.array_equal(initial_dual_lengths < 0, edge_folds(tangled_topology, directions, even_weights) < 0)
assert crossed_counts[0] > 0 and crossed_counts[-1] == 0 and (np.diff(crossed_counts) <= 0).all()
assert (final_dual_lengths > 0).all()
assert (pole_sides(*rescued_history[-1]) > 0).all()
assert bool((max_margin > 0).all())